In [1]:
import pandas as pd
from pathlib import Path

ROOT = Path(r"..")
TRAIN_DIR = ROOT / "data" / "processed" / "train_test"

X_train = pd.read_csv(TRAIN_DIR / "X_train.csv")
X_test = pd.read_csv(TRAIN_DIR / "X_test.csv")

y_train = pd.read_csv(TRAIN_DIR / "y_train.csv").squeeze()
y_test = pd.read_csv(TRAIN_DIR / "y_test.csv").squeeze()

print(X_train.shape)
print(X_test.shape)

(556, 9)
(140, 9)


In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Logistic Regression needs scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 1. Logistic Regression
lr = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr.fit(X_train_scaled, y_train)

# 2. Random Forest
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# 3. XGBoost
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train)

print("✅ All three models trained.")

✅ All three models trained.


In [3]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

models = {
    "Logistic Regression": (lr, X_test_scaled),
    "Random Forest": (rf, X_test),
    "XGBoost": (xgb, X_test)
}

results = []

for name, (model, X_eval) in models.items():

    pred = model.predict(X_eval)
    prob = model.predict_proba(X_eval)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, prob)
    })

results_df = pd.DataFrame(results)

results_df

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.892857,0.950820,0.828571,0.885496,0.957755
1,Random Forest,0.892857,0.910448,0.871429,0.890511,0.964184
2,XGBoost,0.892857,0.910448,0.871429,0.890511,0.954286


In [4]:
results_df.round(4)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.8929,0.9508,0.8286,0.8855,0.9578
1,Random Forest,0.8929,0.9104,0.8714,0.8905,0.9642
2,XGBoost,0.8929,0.9104,0.8714,0.8905,0.9543


In [5]:
results_df.to_csv(
    ROOT / "data" / "processed" / "model_comparison.csv",
    index=False
)

print("✅ Model comparison saved.")

✅ Model comparison saved.
